# Spot-check: MI Large Office — Monthly Energy Totals

Loads the two raw CSV files for Washtenaw County, MI — Large Office, upgrade 0 (Baseline)
and upgrade 36 (Package 3) — and uses DuckDB to spot-check total energy usage by month.

**Source files**
```
data/raw/MI_G2601610/upgrade_0/up00-g2601610-largeoffice.csv
data/raw/MI_G2601610/upgrade_36/up36-g2601610-largeoffice.csv
```

**Checks performed**
1. Row count and column inventory per file
2. `floor_area_represented` — confirm constant, non-null, non-zero
3. Monthly aggregate kWh — site energy, electricity, natural gas
4. Monthly totals normalised to kWh / 1,000 sqft — baseline vs Package 3 side-by-side
5. Fuel-mix check: electricity + gas + other fuels ≈ site total (per row)
6. Null and negative value counts per energy column
7. Interval completeness — confirm 35,040 rows, no gaps

In [1]:
import os
import duckdb
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT_DIR = os.path.normpath(os.path.join(os.getcwd(), ".."))
RAW_DIR  = os.path.join(ROOT_DIR, "data", "raw", "MI_G2601610")

FILE_BASE = os.path.join(RAW_DIR, "upgrade_0",  "up00-g2601610-largeoffice.csv")
FILE_PKG3 = os.path.join(RAW_DIR, "upgrade_36", "up36-g2601610-largeoffice.csv")

for f in [FILE_BASE, FILE_PKG3]:
    assert os.path.exists(f), f"File not found: {f}"
    print(f"Found: {f}")

Found: c:\Users\bbieber\GitHub\terra.do_studio\data\raw\MI_G2601610\upgrade_0\up00-g2601610-largeoffice.csv
Found: c:\Users\bbieber\GitHub\terra.do_studio\data\raw\MI_G2601610\upgrade_36\up36-g2601610-largeoffice.csv


## 1. Row count and column inventory

In [2]:
con = duckdb.connect()

for label, path in [("Baseline (up00)", FILE_BASE), ("Package 3 (up36)", FILE_PKG3)]:
    n = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    cols = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    print(f"\n{label}")
    print(f"  Rows : {n:,}   (expected 35,040 = 96 intervals/day × 365 days)")
    print(f"  Cols : {len(cols)}")
    print(cols[["column_name", "column_type"]].to_string(index=False))


Baseline (up00)
  Rows : 35,040   (expected 35,040 = 96 intervals/day × 365 days)
  Cols : 31
                                              column_name column_type
                                                  upgrade      BIGINT
                                                in.county     VARCHAR
                                in.comstock_building_type     VARCHAR
                                                timestamp   TIMESTAMP
                                              models_used      BIGINT
                                   floor_area_represented      DOUBLE
      out.district_cooling.cooling.energy_consumption.kwh      DOUBLE
      out.district_heating.heating.energy_consumption.kwh      DOUBLE
out.district_heating.water_systems.energy_consumption.kwh      DOUBLE
           out.electricity.cooling.energy_consumption.kwh      DOUBLE
 out.electricity.exterior_lighting.energy_consumption.kwh      DOUBLE
              out.electricity.fans.energy_consumption.kwh      DO


Package 3 (up36)
  Rows : 35,040   (expected 35,040 = 96 intervals/day × 365 days)
  Cols : 56
                                                      column_name column_type
                                                          upgrade      BIGINT
                                                        in.county     VARCHAR
                                        in.comstock_building_type     VARCHAR
                                                        timestamp   TIMESTAMP
                                                      models_used      BIGINT
                                           floor_area_represented      DOUBLE
              out.district_cooling.cooling.energy_consumption.kwh      DOUBLE
              out.district_heating.heating.energy_consumption.kwh      DOUBLE
        out.district_heating.water_systems.energy_consumption.kwh      DOUBLE
                   out.electricity.cooling.energy_consumption.kwh      DOUBLE
         out.electricity.exterior_lighting.ene

## 2. `floor_area_represented` — must be constant, non-null, non-zero

In [3]:
for label, path in [("Baseline", FILE_BASE), ("Package 3", FILE_PKG3)]:
    result = con.execute(f"""
        SELECT
            COUNT(*)                               AS total_rows,
            COUNT(floor_area_represented)          AS non_null_rows,
            COUNT_IF(floor_area_represented = 0)   AS zero_rows,
            COUNT(DISTINCT floor_area_represented) AS distinct_values,
            MIN(floor_area_represented)            AS min_sqft,
            MAX(floor_area_represented)            AS max_sqft
        FROM read_csv_auto('{path}')
    """).df()
    print(f"{label}:")
    display(result)

fa = con.execute(
    f"SELECT floor_area_represented FROM read_csv_auto('{FILE_BASE}') LIMIT 1"
).fetchone()[0]
print(f"floor_area_represented = {fa:,.0f} sqft  ({fa/1e6:.2f} M sqft)")

Baseline:


,total_rows,non_null_rows,zero_rows,distinct_values,min_sqft,max_sqft
0,35040,35040,0.0,1,9.152990e+06,9.152990e+06


Package 3:


,total_rows,non_null_rows,zero_rows,distinct_values,min_sqft,max_sqft
0,35040,35040,0.0,1,9.152990e+06,9.152990e+06


floor_area_represented = 9,152,990 sqft  (9.15 M sqft)


## 3. Monthly aggregate kWh — raw (sum across all buildings)

In [4]:
# DuckDB auto-detects timestamp column as TIMESTAMP — use MONTH() directly
monthly_sql = """
SELECT
    MONTH(timestamp)                                                           AS month,
    ROUND(SUM("out.site_energy.total.energy_consumption.kwh"),          0)    AS site_total_kwh,
    ROUND(SUM("out.electricity.total.energy_consumption.kwh"),          0)    AS elec_total_kwh,
    ROUND(SUM("out.natural_gas.total.energy_consumption.kwh"),          0)    AS gas_total_kwh,
    ROUND(SUM("out.electricity.heating.energy_consumption.kwh"),        0)    AS elec_heating_kwh,
    ROUND(SUM("out.electricity.cooling.energy_consumption.kwh"),        0)    AS elec_cooling_kwh,
    COUNT(*)                                                                   AS n_rows
FROM read_csv_auto('{path}')
GROUP BY MONTH(timestamp)
ORDER BY month
"""

print("=== Baseline (upgrade 0) — raw kWh by month ===")
base_monthly = con.execute(monthly_sql.format(path=FILE_BASE)).df()
display(base_monthly)

print("=== Package 3 (upgrade 36) — raw kWh by month ===")
pkg3_monthly = con.execute(monthly_sql.format(path=FILE_PKG3)).df()
display(pkg3_monthly)

=== Baseline (upgrade 0) — raw kWh by month ===


,month,site_total_kwh,elec_total_kwh,gas_total_kwh,elec_heating_kwh,elec_cooling_kwh,n_rows
0,1,19014334.0,8869203.0,9525114.0,2069587.0,116372.0,2976
1,2,13946013.0,7479753.0,5935048.0,1258373.0,233233.0,2688
2,3,13017147.0,7812721.0,4873123.0,1101462.0,132352.0,2976
3,4,10792931.0,7500867.0,3064938.0,654854.0,504633.0,2880
4,5,10485598.0,9842956.0,353633.0,21328.0,2715732.0,2976
5,6,10744683.0,10207251.0,191115.0,1272.0,3282952.0,2880
6,7,11602687.0,11040504.0,140393.0,148.0,3799401.0,2976
7,8,11756662.0,11191022.0,149987.0,59.0,3882550.0,2976
8,9,10470415.0,9813868.0,310996.0,8193.0,2996221.0,2880
9,10,10350881.0,8434103.0,1652187.0,256765.0,1314208.0,2976


=== Package 3 (upgrade 36) — raw kWh by month ===


,month,site_total_kwh,elec_total_kwh,gas_total_kwh,elec_heating_kwh,elec_cooling_kwh,n_rows
0,1,13256404.0,12385097.0,224027.0,6138692.0,112662.0,2976
1,2,9934044.0,9246281.0,138935.0,3520649.0,227110.0,2688
2,3,9539257.0,9059742.0,124882.0,2893322.0,126009.0,2976
3,4,8379035.0,8046317.0,92089.0,1749454.0,488658.0,2880
4,5,9446186.0,9115332.0,55051.0,119634.0,2630666.0,2976
5,6,9812322.0,9429933.0,51421.0,46352.0,3187492.0,2880
6,7,10588878.0,10133512.0,52049.0,28985.0,3670546.0,2976
7,8,10715636.0,10266549.0,52044.0,31759.0,3757216.0,2976
8,9,9514052.0,9130469.0,50159.0,89453.0,2909901.0,2880
9,10,8644130.0,8310370.0,62372.0,805440.0,1273159.0,2976


## 4. Monthly totals normalised to kWh / 1,000 sqft — Baseline vs Package 3

In [5]:
MONTH_NAMES = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",  5: "May",  6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

norm_sql = """
SELECT
    MONTH(timestamp) AS month,
    ROUND(SUM("out.site_energy.total.energy_consumption.kwh")
          / (MAX(floor_area_represented) / 1000.0), 3)  AS site_kwh_per_1000sqft,
    ROUND(SUM("out.electricity.total.energy_consumption.kwh")
          / (MAX(floor_area_represented) / 1000.0), 3)  AS elec_kwh_per_1000sqft,
    ROUND(SUM("out.natural_gas.total.energy_consumption.kwh")
          / (MAX(floor_area_represented) / 1000.0), 3)  AS gas_kwh_per_1000sqft,
    ROUND(SUM("out.electricity.heating.energy_consumption.kwh")
          / (MAX(floor_area_represented) / 1000.0), 4)  AS elec_heating_kwh_per_1000sqft,
    ROUND(SUM("out.electricity.cooling.energy_consumption.kwh")
          / (MAX(floor_area_represented) / 1000.0), 4)  AS elec_cooling_kwh_per_1000sqft
FROM read_csv_auto('{path}')
GROUP BY MONTH(timestamp)
ORDER BY month
"""

base_norm = con.execute(norm_sql.format(path=FILE_BASE)).df()
pkg3_norm = con.execute(norm_sql.format(path=FILE_PKG3)).df()

cmp = base_norm.merge(pkg3_norm, on="month", suffixes=("_base", "_pkg3"))
cmp["month_name"] = cmp["month"].map(MONTH_NAMES)
cmp["site_pct_change"] = (
    (cmp["site_kwh_per_1000sqft_pkg3"] - cmp["site_kwh_per_1000sqft_base"])
    / cmp["site_kwh_per_1000sqft_base"] * 100
).round(1)

display(cmp[[
    "month_name",
    "site_kwh_per_1000sqft_base", "site_kwh_per_1000sqft_pkg3", "site_pct_change",
    "elec_kwh_per_1000sqft_base", "elec_kwh_per_1000sqft_pkg3",
    "gas_kwh_per_1000sqft_base",  "gas_kwh_per_1000sqft_pkg3",
    "elec_heating_kwh_per_1000sqft_base", "elec_heating_kwh_per_1000sqft_pkg3",
]])

,month_name,site_kwh_per_1000sqft_base,site_kwh_per_1000sqft_pkg3,site_pct_change,elec_kwh_per_1000sqft_base,elec_kwh_per_1000sqft_pkg3,gas_kwh_per_1000sqft_base,gas_kwh_per_1000sqft_pkg3,elec_heating_kwh_per_1000sqft_base,elec_heating_kwh_per_1000sqft_pkg3
0,Jan,2077.390,1448.314,-30.3,968.995,1353.120,1040.656,24.476,226.1105,670.6761
1,Feb,1523.657,1085.333,-28.8,817.192,1010.192,648.427,15.179,137.4822,384.6447
2,Mar,1422.174,1042.201,-26.7,853.570,989.812,532.408,13.644,120.3390,316.1067
3,Apr,1179.170,915.442,-22.4,819.499,879.092,334.856,10.061,71.5454,191.1347
4,May,1145.593,1032.033,-9.9,1075.381,995.886,38.636,6.015,2.3302,13.0705
5,Jun,1173.899,1072.035,-8.7,1115.182,1030.257,20.880,5.618,0.1390,5.0642
6,Jul,1267.639,1156.876,-8.7,1206.218,1107.126,15.338,5.687,0.0162,3.1667
7,Aug,1284.461,1170.725,-8.9,1222.663,1121.661,16.387,5.686,0.0064,3.4698
8,Sep,1143.934,1039.447,-9.1,1072.203,997.539,33.978,5.480,0.8951,9.7731
9,Oct,1130.874,944.405,-16.5,921.459,907.940,180.508,6.814,28.0526,87.9975


## 5. Fuel-mix check — component fuel totals should equal site energy total

Per-row check: `electricity + natural_gas + other_fuel + district_cooling + district_heating ≈ site_total`  
Rows with absolute difference > 0.1 kWh are flagged.

In [6]:
fuel_check_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(
        ABS(
              "out.electricity.total.energy_consumption.kwh"
            + "out.natural_gas.total.energy_consumption.kwh"
            + "out.other_fuel.total.energy_consumption.kwh"
            + "out.district_cooling.total.energy_consumption.kwh"
            + "out.district_heating.total.energy_consumption.kwh"
            - "out.site_energy.total.energy_consumption.kwh"
        ) > 0.1
    ) AS rows_with_mismatch,
    ROUND(MAX(
        ABS(
              "out.electricity.total.energy_consumption.kwh"
            + "out.natural_gas.total.energy_consumption.kwh"
            + "out.other_fuel.total.energy_consumption.kwh"
            + "out.district_cooling.total.energy_consumption.kwh"
            + "out.district_heating.total.energy_consumption.kwh"
            - "out.site_energy.total.energy_consumption.kwh"
        )
    ), 6) AS max_abs_diff_kwh
FROM read_csv_auto('{path}')
"""

for label, path in [("Baseline", FILE_BASE), ("Package 3", FILE_PKG3)]:
    result = con.execute(fuel_check_sql.format(path=path)).df()
    print(f"{label}:")
    display(result)

Baseline:


,total_rows,rows_with_mismatch,max_abs_diff_kwh
0,35040,0.0,0.0


Package 3:


,total_rows,rows_with_mismatch,max_abs_diff_kwh
0,35040,0.0,0.0


## 6. Null and negative value counts per energy column

In [7]:
cols_df = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{FILE_BASE}')").df()
energy_cols = [
    c for c in cols_df["column_name"]
    if c.startswith("out.") and not c.endswith(".savings")
]

for label, path in [("Baseline", FILE_BASE), ("Package 3", FILE_PKG3)]:
    rows = []
    for c in energy_cols:
        r = con.execute(f"""
            SELECT
                '{c}' AS column_name,
                COUNT_IF("{c}" IS NULL) AS null_count,
                COUNT_IF("{c}" < 0)    AS negative_count,
                ROUND(MIN("{c}"), 4)   AS min_value,
                ROUND(MAX("{c}"), 4)   AS max_value
            FROM read_csv_auto('{path}')
        """).df()
        rows.append(r)
    summary = pd.concat(rows, ignore_index=True)
    flagged = summary[(summary["null_count"] > 0) | (summary["negative_count"] > 0)]
    print(f"\n{label} — {len(energy_cols)} energy columns checked:")
    if flagged.empty:
        print("  No nulls or negative values found.")
    else:
        display(flagged)


Baseline — 25 energy columns checked:
  No nulls or negative values found.



Package 3 — 25 energy columns checked:
  No nulls or negative values found.


## 7. Interval completeness — confirm 35,040 rows and no 15-minute gaps

In [8]:
interval_sql = """
WITH ordered AS (
    SELECT
        timestamp,
        LAG(timestamp) OVER (ORDER BY timestamp) AS prev_ts
    FROM read_csv_auto('{path}')
)
SELECT
    COUNT(*)                                                    AS total_rows,
    COUNT(DISTINCT timestamp)                                   AS distinct_timestamps,
    MIN(timestamp)                                              AS first_timestamp,
    MAX(timestamp)                                              AS last_timestamp,
    COUNT_IF(
        prev_ts IS NOT NULL
        AND epoch(timestamp) - epoch(prev_ts) != 900
    )                                                           AS non_15min_gaps
FROM ordered
"""

for label, path in [("Baseline", FILE_BASE), ("Package 3", FILE_PKG3)]:
    result = con.execute(interval_sql.format(path=path)).df()
    print(f"{label}:")
    display(result)

con.close()
print("Done.")

Baseline:


,total_rows,distinct_timestamps,first_timestamp,last_timestamp,non_15min_gaps
0,35040,35040,2018-01-01 00:15:00,2019-01-01,0.0


Package 3:


,total_rows,distinct_timestamps,first_timestamp,last_timestamp,non_15min_gaps
0,35040,35040,2018-01-01 00:15:00,2019-01-01,0.0


Done.
